## 1. what are subagents?

### Core Concept
- **Subagents = specialized helpers** that Claude Code delegates tasks to — each runs in its **own isolated context window**, does the work, returns only a **summary** to the main thread.
- All the messy intermediate work (file reads, searches, tool calls) **stays isolated** and never pollutes the main conversation.

### Why They Matter （main context window VS subagent context window）
- Main context window is **finite** — every tool call, file read, and search result fills it up.
- Once full, Claude **loses track** of earlier parts of the conversation.
- Subagents solve this by spinning up a **separate context window** that gets discarded after the task.

### How They Work
A subagent receives **two inputs**:
1. **Custom system prompt** — defines the subagent's role and behavior (from config file)
2. **Task description** — written by the parent agent based on your request

The subagent works independently, then returns **only a summary** to the main conversation. The entire subagent context is then **thrown away**.

### Tradeoff
- ✅ Main context stays clean — you get the answer without the noise of the journey
- ❌ You lose **visibility** into how the subagent reached its conclusions

### Concrete Example
Question: *"Which service handles refunds?"*

| Without subagent | With subagent |
|---|---|
| Claude reads 15 files, runs searches, traces functions — **all of it fills your main context** | Explore subagent does the digging in its own context — **main context only records the question + the answer** |

### Built-in Subagents
- **General purpose** — multi-step tasks needing both exploration and action
- **Explore** — fast searching and navigation of codebases
- **Plan** — research and analysis during plan mode, before presenting a plan

### Custom Subagents
You can define your own with custom system prompts and tool access — code reviewer, test writer, doc generator, anything.

---

### 💡 Three Key Benefits
1. **Focus** — each subagent concentrates on one specific task
2. **Clean context** — intermediate work is isolated, main thread stays uncluttered
3. **Concise output** — only the summary you actually need comes back

### One-line Takeaway
> Subagents = isolated context windows that do dirty work elsewhere and report back only the summary. The less noise in the main context, the longer and more effectively you can work.




## 2. Creating a subagent?
### 2.1 Subagent formating 
 They are defined as markdown files with YAML frontmatter that tell Claude when to use the subagent and how the subagent should behave 
 - Name
 - Description
 - Tools
 - Model
 - Color

 ### 2.2 Creating a subagent
 #### The command to create a subagent
The easiest way to create a subagent is with the **/agents** slash command. This opens the main interface for managing your subagents. From there, select Create new agent.

#### Choose the scope of the subagent
- Project-level -- available only in the current project
- User-level -- shared across all projects on your machine

#### Two ways to create the subagent
1. Mannual Configuration 
2. Let Claude generate it (Recommended)

#### Just describe what you want the subagent to do
e.g. "Create a code-reviewer that reviews code changes for me."

#### Customizing Tools
During creation, you get the chance to customize which tools the subagent can access. The tool categories include:

- Read-only tools
- Edit tools
- Execution tools
- MCP tools
- Other tools

#### Choosing the model 
- Haiku -- best for fast, lightweight tasks
- Sonnet -- a good middle ground between speed and depth
- Opus -- best for complex analysis
- Inherit -- uses whatever model your main conversation is running


#### Check the Config File
check "/Users/ionahu/sources/NomNom/.claude/agents/code-reviewer.md"

- name: code-quality-reviewer

- description: Use this agent when you need to review recently written or modified code for quality, security, and best practice compliance.This is how the main agent decides which subagent to launch and when!!!

- tools: Bash, Glob, Grep, Read, WebFetch, WebSearch

- model: sonnet

- Color: purple

- Skills

- **System Prompts**
    The body of the markdown file (everything below the YAML frontmatter) is the system prompt. This is where you give the subagent its instructions: what it should focus on, how it should analyze things, and how it should report findings back to the main agent.

    A well-written system prompt is the difference between a useful subagent and one that misses the point. Be specific about what the subagent should look for and how it should structure its output.


### 2.3 Making Claude Use Your Subagent Automatically
If you want Claude to delegate tasks to the subagent without you explicitly asking, include the word "**proactively**" in the description field. For example:

description: Proactively suggest running this agent after major code changes...

### 2.4 Testing your subagent 
check if the subagent works

### 3.1 How Subagent Config Data Gets Used
When you send a message to the main context window agent, the name and description of every available subagent are included in the system prompt. The description of the config file plays two main roles:

- This is how the main agent decides which subagent to launch and when. 
- The description doesn't just control when a subagent runs -- it shapes what the subagent is told to do.

### 3.2 Four mian components that improve a subagent's effectiveness
Effective subagents share four characteristics:

1. **Specific descriptions** -- The description controls when the subagent is launched and what instructions it receives. Write it to steer both.

    e.g. Replace a generic description of Code reviewer with "You must tell the agent precisely which files you want it to review". 

    With a generic description, the main agent might write an input prompt like "use get diff to find the current changes." That's vague. The subagent has to figure out which files matter on its own.

    If you update the description to include something like "You must tell the agent precisely which files you want it to review," the main agent will now write a much more specific input prompt that lists the actual files to review.


2. **Structured output** -- Define an output format in the system prompt so the subagent knows when it's done and returns information the main thread can use.

3. **Obstacle reporting** -- Include a section in the output format for workarounds, quirks, and problems so the main thread doesn't have to rediscover them.

4. **Limited tool access** -- Only give a subagent the tools it actually needs. Read-only for research, bash for reviewers, edit/write only for agents that should change code.

    Each of these patterns is simple on its own, but together they turn a subagent from something that vaguely tries to help into a focused, predictable worker that finishes on time and reports back clearly.



---

## In summary - Designing an effective subagent

I just learned how to create a subagent, but creating ≠ effective — new subagents typically have three problems: they wander (unclear what to do), run too long (no idea when to stop), and return output the main agent can't use. Before fixing these, understand one key insight: the description field is used by the main Claude twice — it decides both when to launch the subagent and what to tell it once launched. So editing the description isn't just changing "when it gets called," it's also changing "what instructions it receives." There are four concrete fixes: (1) Make the description directive — e.g., add "you must tell the agent..." and the main Claude will automatically bake those requirements into the kickoff prompt; (2) Define an output format in the system prompt — the single most important fix; give the subagent a fill-in-the-blanks template so it knows it's done once every section is filled, instead of running indefinitely; (3) Make the subagent report obstacles — when it returns, beyond saying "task done," it should also surface "what I struggled with and how I worked around it," so lessons stick and the main thread doesn't have to rediscover them; (4) Minimum tool access — only give a subagent the tools it actually needs (Read/Grep for research, plus Bash for reviewers, Edit/Write only for agents that modify code), which prevents unintended side effects and keeps each subagent's role crystal clear.

---


## 4. Using subagents effectively

### Core Decision Rule

Before reaching for a subagent, ask yourself one question: **Do I need to see the intermediate work?**

- **No** → Use a subagent (it hides the process and returns only a summary)
- **Yes** → Keep it in the main thread (any detail that goes into a subagent gets lost)

This is the **single criterion** for deciding when to use a subagent. Every scenario below is just an application of this rule.

---

### ✅ Three Scenarios Where Subagents Shine

**1. Research / Exploration Tasks**
You only want the answer, not "the 30 files I read along the way." Example: figuring out where JWT validation happens in an unfamiliar codebase → the subagent returns one line: *"validated in `auth.js:42`"*.

**2. Code Reviews**
The key benefit isn't context isolation — it's **psychological isolation**. Claude tends to go easy on code it helped write. A reviewer subagent sees the code in a fresh context, **doesn't know Claude wrote it**, and reviews it like a stranger would — much more rigorously. Bonus: you can encode team-specific review standards in its system prompt.

**3. Tasks Needing a Non-Default Style**
Claude Code's default system prompt is optimized for writing code (concise, technical). Tasks like copywriting or styling need the opposite — give a copywriting subagent a completely different prompt (warm, persuasive); give a styling subagent auto-loaded design system files so it knows your color tokens and spacing rules upfront.

---

### ❌ Three Anti-Patterns to Avoid

**1. "Expert" Personas**
Writing a description like *"You are a Python expert"* adds nothing — Claude already knows Python; slapping a title on it doesn't make it smarter. A subagent's value comes from providing something genuinely unique (isolated context, custom instructions, restricted tools). A pure persona delivers none of those.

**2. Sequential Pipelines** ⭐ Critical
For example, three subagents in a row: "reproduce bug → debug → fix." Why it fails: subagents pass information through **summaries**, and a lot of detail gets dropped along the way. Subagent A's discovery that *"the bug only triggers with special characters"* never makes it into the summary, so B has to rediscover it — and C's fix might miss the point entirely.

**Rule**: Steps that are **independent** → subagent is fine. Steps that **depend on each other's findings** (debugging, iterative development) → keep them in the main thread.

**3. Test Runners**
When tests fail, you need the **full output** to diagnose — traceback, assertion details, stderr. A subagent will compress all of that into "3 tests failed," leaving you unable to figure out what actually broke.

> Anthropic's own testing: **the test-runner pattern performed the worst of all subagent configurations tested.**

---

### 💡 Three Questions to Build the Habit

Each time you're tempted to use a subagent, walk through these:

1. **Does this need a "fresh-eyes" perspective or a non-default prompt?** → Yes = strong signal to use one
2. **Does each step depend on the previous step's findings?** → Yes = don't use one
3. **Will I need to see the full intermediate output if something fails?** → Yes = don't use one

---

### One-Line Takeaway
> A subagent's whole purpose is to **hide the process and serve only the conclusion**. 

